# Multi-Fish Cohort `[56h]` + `[56g]`

This notebook aggregates already processed fish and produces cohort-level versions of:
- `[56h]` (BPI + condensed ipsi/contra traces)
- `[56g]` (BPI vs activity diagnostics)

Configured fish for now:
- `L395_f11` (`owner='Danin'`)
- `L396_f01` (`owner='Matilde'`)
- `L396_f04` (`owner='Matilde'`)


In [ ]:
# [cfg] Imports + cohort configuration
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.measure import regionprops_table
from IPython.display import display


def _default_nas_root():
    if os.name == "nt":
        base = Path(r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c")
        return base / "07_Data" if (base / "07_Data").exists() else base
    return Path("/Volumes/jlarsch/default/D2c/07_Data")


NAS_ROOT = Path(globals().get("NAS_ROOT", _default_nas_root()))
MATCHING_METADATA_CSV = Path(globals().get(
    "MATCHING_METADATA_CSV",
    str(NAS_ROOT / "Danin" / "matchingMetadata.csv"),
))

FISH_SPECS = list(globals().get("FISH_SPECS", [
    {"owner": "Danin", "fish_id": "L395_f11"},
    {"owner": "Matilde", "fish_id": "L396_f01"},
    {"owner": "Matilde", "fish_id": "L396_f04"},
]))

# Core timing/trace parameters (aligned with single-fish [56h]).
WINDOW_PRE_SEC = float(globals().get("WINDOW_PRE_SEC", 20.0))
WINDOW_POST_SEC = float(globals().get("WINDOW_POST_SEC", 50.0))
STIM_ONSET_DELAY_SEC = float(globals().get("STIM_ONSET_DELAY_SEC", 10.0))
WINDOW_EDGE_POLICY = str(globals().get("WINDOW_EDGE_POLICY", "pad_nan")).strip().lower()
MIN_VALID_POINTS_PER_SEG = int(globals().get("MIN_VALID_POINTS_PER_SEG", 1))
BPI_EDGE_POLICY = str(globals().get("BPI_EDGE_POLICY", "pad_nan")).strip().lower()
BPI_MIN_VALID_FRAC = float(globals().get("BPI_MIN_VALID_FRAC", 0.5))
BPI_MIN_TRIALS_PER_CLASS = int(globals().get("BPI_MIN_TRIALS_PER_CLASS", 3))
BPI_DENOM_EPS = float(globals().get("BPI_DENOM_EPS", 1e-6))
ZSCORE_MIN_BASELINE_POINTS = int(globals().get("ZSCORE_MIN_BASELINE_POINTS", 200))
ZSCORE_MIN_BASELINE_STD = float(globals().get("ZSCORE_MIN_BASELINE_STD", 1e-6))
STIM_TIME_SCALE = float(globals().get("STIM_TIME_SCALE", 1.0))

MEASURE_START_BLOCK = globals().get("MEASURE_START_BLOCK", 1)
MEASURE_START_EVENT = str(globals().get("MEASURE_START_EVENT", "start"))
REMOVE_INTERBLOCK_GAPS = bool(globals().get("REMOVE_INTERBLOCK_GAPS", True))

# Cohort plotting parameters.
MIN_SEGMENTS = int(globals().get("MIN_SEGMENTS", 3))
MIN_CELLS = int(globals().get("MIN_CELLS", 1))
MAX_GENES = int(globals().get("MAX_GENES", 12))
USE_SEM = bool(globals().get("USE_SEM", True))
COHORT_TRACE_DT = globals().get("COHORT_TRACE_DT", None)  # seconds; None -> infer from first fish fps

# Optional high-confidence filter if low-confidence columns are present in conf_to_func_pairs.csv.
USE_HIGH_CONF_ONLY = bool(globals().get("USE_HIGH_CONF_ONLY", True))
LOW_CONF_COLUMNS = list(globals().get(
    "LOW_CONF_COLUMNS",
    ["_is_low_conf_q95", "_is_low_conf_match_any_globalR50", "is_low_confidence_segmentation"],
))

DEFAULT_GENE_ORDER = ['sst1.1', 'sst1.2', 'npy', 'tac3b', 'pth2', 'cfos', 'cort']
DEFAULT_GENE_COLORS = {
    'sst1.1': '#d62728',
    'sst1.2': '#d61ad2',
    'npy': '#1f9d55',
    'tac3b': '#ffd400',
    'pth2': '#00bcd4',
    'cfos': '#ff7f0e',
    'cort': '#8c564b',
}
GENE_ORDER = list(globals().get('GENE_ORDER', DEFAULT_GENE_ORDER))
GENE_COLORS = dict(DEFAULT_GENE_COLORS)
try:
    _gc = globals().get('GENE_COLORS', {})
    if isinstance(_gc, dict):
        GENE_COLORS.update(_gc)
except Exception:
    pass

PANEL_GRID = [["contra_LB", "contra_LC"], ["ipsi_LB", "ipsi_LC"]]
PLOT_ORDER = [p for row in PANEL_GRID for p in row]
PLOT_TITLES = {
    "contra_LB": "Contra × bout-like",
    "contra_LC": "Contra × continuous",
    "ipsi_LB": "Ipsi × bout-like",
    "ipsi_LC": "Ipsi × continuous",
}

COHORT_OUTDIR = Path(globals().get(
    "COHORT_OUTDIR",
    str(Path.cwd() / "cohort_outputs" / "multi_fish_56h_56g"),
))
COHORT_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"[cfg] NAS_ROOT={NAS_ROOT}")
print(f"[cfg] MATCHING_METADATA_CSV={MATCHING_METADATA_CSV}")
print(f"[cfg] fishes={[(d['owner'], d['fish_id']) for d in FISH_SPECS]}")
print(f"[cfg] COHORT_OUTDIR={COHORT_OUTDIR}")


In [ ]:
# [helpers] File discovery, parsing, side-assignment, and trace utilities
import pathlib


def owner_root(nas_root, owner):
    base = Path(nas_root) / str(owner)
    mic = base / "Microscopy"
    return mic if mic.exists() else base


def fish_paths(nas_root, owner, fish_id):
    fish_dir = owner_root(nas_root, owner) / str(fish_id)
    analysis_dir = fish_dir / "03_analysis"
    func_dir = analysis_dir / "functional"
    out_reg = func_dir / "registration"
    suite2p_root = func_dir / "suite2P"
    conf_csv = out_reg / "conf_to_func_pairs.csv"
    midline_json = out_reg / "midline_params_func_ref.json"
    metadata_dir = fish_dir / "01_raw" / "2p" / "metadata"
    return {
        "fish_dir": fish_dir,
        "analysis_dir": analysis_dir,
        "func_dir": func_dir,
        "out_reg": out_reg,
        "suite2p_root": suite2p_root,
        "conf_csv": conf_csv,
        "midline_json": midline_json,
        "metadata_dir": metadata_dir,
    }


def load_matching_metadata(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return pd.read_csv(p)
    except Exception:
        return None


_MD_CACHE = load_matching_metadata(MATCHING_METADATA_CSV)


def get_polarity(fish_id, md_df=None):
    if md_df is None:
        md_df = _MD_CACHE
    if md_df is None or md_df.empty or "fish_id" not in md_df.columns:
        return None
    row = md_df.loc[md_df["fish_id"].astype(str) == str(fish_id)]
    if row.empty:
        return None
    pol = str(row.iloc[0].get("polarity", "")).strip().lower()
    if pol in {"north", "south"}:
        return pol
    return None


def _block_key(code):
    m = re.match(r"^B(\d+)$", str(code))
    return int(m.group(1)) if m else 10**9


def find_experiment_log(metadata_dir, fish_id):
    base = Path(metadata_dir)
    if not base.exists():
        return None
    pats = [f"*{fish_id}*experiment_log*.csv", "*experiment_log*.csv"]
    hits = []
    for pat in pats:
        hits.extend(sorted(base.glob(pat)))
    if not hits:
        return None
    hits = sorted(set(hits), key=lambda p: p.stat().st_mtime)
    return hits[-1]


def load_events_df(csv_path, time_scale=1.0):
    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError("experiment log is empty")
    cols = {c: c.strip().lower() for c in df.columns}
    df = df.rename(columns=cols)

    event_cols = [c for c in df.columns if "event" in c]
    time_cols = [c for c in df.columns if ("timestamp" in c) or (c == "time")]

    if "event" in df.columns:
        event_col = "event"
    elif event_cols:
        event_col = event_cols[0]
    else:
        raise ValueError(f"Could not infer event column from {list(df.columns)}")

    if "timestamp" in df.columns:
        time_col = "timestamp"
    elif "time" in df.columns:
        time_col = "time"
    elif time_cols:
        time_col = time_cols[0]
    else:
        raise ValueError(f"Could not infer time column from {list(df.columns)}")

    out = df[[event_col, time_col]].rename(columns={event_col: "event", time_col: "time"})
    out["event"] = out["event"].astype(str).str.strip()
    out["time"] = pd.to_numeric(out["time"], errors="coerce") * float(time_scale)
    out = out.dropna(subset=["event", "time"]).sort_values("time").reset_index(drop=True)
    return out


def build_stim_table(df_evt, measure_start_block=1, measure_start_event="start", remove_interblock_gaps=True):
    evt = df_evt.copy()

    if measure_start_block is not None:
        if isinstance(measure_start_block, str):
            block_label = measure_start_block
        else:
            block_label = f"B{int(measure_start_block)}"
        start_event = f"{block_label}_{measure_start_event}"
        start_match = evt.loc[evt["event"] == start_event, "time"]
        if len(start_match):
            t0 = float(start_match.iloc[0])
        else:
            block_rows = evt[evt["event"].str.startswith(f"{block_label}_")]
            t0 = float(block_rows["time"].min()) if not block_rows.empty else None
        if t0 is not None:
            evt["time"] = evt["time"] - t0
            evt = evt[evt["time"] >= 0].reset_index(drop=True)

    block_codes = []
    for ev in evt["event"]:
        m = re.match(r"^(B\d+)_", ev)
        if m:
            block_codes.append(m.group(1))
    block_codes = sorted(set(block_codes), key=_block_key)

    blocks = []
    for b in block_codes:
        block_events = evt[evt["event"].str.startswith(f"{b}_")]
        if block_events.empty:
            continue
        bstart = None
        bend = None
        s = block_events.loc[block_events["event"] == f"{b}_start", "time"]
        if len(s):
            bstart = float(s.iloc[0])
        e = block_events.loc[block_events["event"] == f"{b}_end", "time"]
        if len(e):
            bend = float(e.iloc[0])
        if bstart is None:
            bstart = float(block_events["time"].min())
        ib = block_events.loc[block_events["event"] == f"{b}_interblock_pause", "time"]
        ib_time = float(ib.iloc[0]) if len(ib) else None
        if bend is None:
            bend = ib_time
        elif ib_time is not None:
            bend = ib_time
        if bend is None:
            bend = float(block_events["time"].max())
        blocks.append({"block": b, "start": bstart, "end": bend})

    if remove_interblock_gaps and blocks:
        blocks = sorted(blocks, key=lambda d: d["start"])
        shift_map = {}
        shift = 0.0
        prev_end = None
        for blk in blocks:
            orig_start = float(blk["start"])
            orig_end = float(blk["end"])
            if prev_end is not None:
                gap = max(0.0, orig_start - prev_end)
                shift += gap
            shift_map[blk["block"]] = shift
            blk["start"] = orig_start - shift
            blk["end"] = orig_end - shift
            prev_end = orig_end

        def _shift_evt_time(row):
            m = re.match(r"^(B\d+)_", str(row["event"]))
            if not m:
                return float(row["time"])
            return float(row["time"]) - float(shift_map.get(m.group(1), 0.0))

        evt["time"] = evt.apply(_shift_evt_time, axis=1)
        evt = evt.sort_values("time").reset_index(drop=True)

    stims = []
    for _, row in evt.iterrows():
        m = re.match(r"^(B\d+)_stim(\d+)_(.+)$", str(row["event"]))
        if not m:
            continue
        block = m.group(1)
        stim_idx = int(m.group(2))
        stim_type = str(m.group(3))
        t0 = float(row["time"])

        end_time = None
        post_name = f"{block}_poststim{stim_idx}_pause"
        match = evt.loc[evt["event"] == post_name, "time"]
        if len(match):
            end_time = float(match.iloc[0])
        if end_time is None:
            after = evt[(evt["time"] > t0) & evt["event"].str.startswith(f"{block}_")].sort_values("time")
            end_time = float(after["time"].iloc[0]) if not after.empty else t0 + 10.0

        stims.append({
            "block": block,
            "stim_idx": stim_idx,
            "type": stim_type,
            "start": t0,
            "end": end_time,
            "duration": float(end_time - t0),
        })

    return evt, pd.DataFrame(stims), blocks


def _load_ops_npy(path):
    try:
        return np.load(path, allow_pickle=True).item()
    except NotImplementedError:
        _orig_win = pathlib.WindowsPath
        _orig_pure = pathlib.PureWindowsPath
        pathlib.WindowsPath = pathlib.PosixPath
        pathlib.PureWindowsPath = pathlib.PurePosixPath
        try:
            return np.load(path, allow_pickle=True).item()
        finally:
            pathlib.WindowsPath = _orig_win
            pathlib.PureWindowsPath = _orig_pure


def _find_suite2p_file(plane_dir, key):
    p = Path(plane_dir) / f"{key}.npy"
    if p.exists():
        return p
    hits = sorted(Path(plane_dir).glob(f"*_{key}.npy"))
    return hits[0] if hits else None


def _plane_num_from_name(name):
    m = re.search(r"plane(\d+)", str(name))
    return int(m.group(1)) if m else None


def apply_func_orientation(arr, polarity):
    if arr is None or getattr(arr, "ndim", 0) < 2:
        return arr
    out = arr
    if str(polarity).lower() == "north":
        out = out[..., ::-1, ::-1]
    out = out[..., ::-1]
    return out


def build_labels_from_stat(stat, iscell, ops):
    Ly = int(ops.get("Ly", 0))
    Lx = int(ops.get("Lx", 0))
    labels = np.zeros((Ly, Lx), dtype=np.uint32)
    keep = np.asarray(iscell)[:, 0].astype(bool)
    idxs = np.where(keep)[0]
    for roi_idx in idxs:
        s = stat[roi_idx]
        if isinstance(s, dict):
            ypix = np.asarray(s.get("ypix", []), dtype=np.int64)
            xpix = np.asarray(s.get("xpix", []), dtype=np.int64)
            overlap = s.get("overlap", None)
        else:
            ypix = np.asarray(s["ypix"], dtype=np.int64)
            xpix = np.asarray(s["xpix"], dtype=np.int64)
            dtype_names = getattr(getattr(s, "dtype", None), "names", None)
            overlap = s["overlap"] if dtype_names and "overlap" in dtype_names else None
        if overlap is not None:
            ok = ~np.asarray(overlap, dtype=bool)
            ypix = ypix[ok]
            xpix = xpix[ok]
        if ypix.size and xpix.size:
            labels[ypix, xpix] = roi_idx + 1
    return labels, keep


def load_suite2p_map(suite2p_root, polarity, dfof_baseline_pct=10.0, dfof_eps=1e-6):
    root = Path(suite2p_root)
    plane_dirs = [p for p in root.glob("plane*") if p.is_dir()]
    plane_dirs = sorted(plane_dirs, key=lambda p: (_plane_num_from_name(p.name) if _plane_num_from_name(p.name) is not None else p.name))

    s2p_map = {}
    roi_rows = []
    fps_vals = []

    for pd_i, plane_dir in enumerate(plane_dirs):
        plane_num = _plane_num_from_name(plane_dir.name)
        if plane_num is None:
            plane_num = int(pd_i)

        paths = {k: _find_suite2p_file(plane_dir, k) for k in ("F", "Fneu", "spks", "stat", "ops", "iscell")}
        if any(v is None for v in paths.values()):
            continue

        F = np.load(paths["F"], allow_pickle=True)
        Fneu = np.load(paths["Fneu"], allow_pickle=True)
        spks = np.load(paths["spks"], allow_pickle=True)
        stat = np.load(paths["stat"], allow_pickle=True)
        ops = _load_ops_npy(paths["ops"])
        iscell = np.load(paths["iscell"], allow_pickle=True)

        labels, keep = build_labels_from_stat(stat, iscell, ops)
        labels = apply_func_orientation(labels, polarity)

        try:
            props = regionprops_table(labels.astype(np.int32, copy=False), properties=("label", "centroid"))
            rdf = pd.DataFrame(props)
            if not rdf.empty:
                rdf = rdf[rdf["label"] != 0].copy()
                if not rdf.empty:
                    rdf = rdf.rename(columns={"label": "func_label", "centroid-0": "y", "centroid-1": "x"})
                    rdf["plane"] = int(plane_num)
                    roi_rows.append(rdf[["plane", "func_label", "x", "y"]])
        except Exception:
            pass

        F_raw = np.asarray(F, dtype=np.float32)
        F0 = np.percentile(F_raw, float(dfof_baseline_pct), axis=1, keepdims=True)
        dff = (F_raw - F0) / (F0 + float(dfof_eps))

        fs = ops.get("fs", None)
        if fs is not None:
            try:
                fps_vals.append(float(fs))
            except Exception:
                pass

        s2p_map[int(plane_num)] = {
            "plane_num": int(plane_num),
            "plane_dir": plane_dir,
            "labels": labels,
            "iscell_keep": keep,
            "F": F,
            "Fneu": Fneu,
            "spks": spks,
            "stat": stat,
            "ops": ops,
            "iscell": iscell,
            "dff": dff,
            "flip_x": False,
            "func_orient": "rot180+flipX" if str(polarity).lower() == "north" else "flipX",
        }

    roi_df = pd.concat(roi_rows, ignore_index=True) if roi_rows else pd.DataFrame(columns=["plane", "func_label", "x", "y"])

    fps = None
    if fps_vals:
        fps = float(fps_vals[0])

    return s2p_map, roi_df, fps


def as_bool_series(s):
    if s is None:
        return pd.Series([], dtype=bool)
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) != 0
    return s.astype(str).str.strip().str.lower().isin({"1", "true", "t", "yes", "y"})


def prepare_pairs_for_analysis(pairs_df):
    req_cols = ["gene", "conf_mask", "conf_label", "anat_label", "func_label", "plane"]
    missing = [c for c in req_cols if c not in pairs_df.columns]
    if missing:
        raise RuntimeError(f"pairs mapping missing required columns: {missing}")

    out = pairs_df.copy()

    if "is_selected_for_analysis" in out.columns:
        sel = as_bool_series(out["is_selected_for_analysis"])
        out = out[sel].copy()

    # Optional high-confidence filtering if low-confidence flags are available.
    if USE_HIGH_CONF_ONLY:
        used_any = False
        for col in LOW_CONF_COLUMNS:
            if col in out.columns:
                used_any = True
                low = as_bool_series(out[col])
                out = out[~low].copy()
        if not used_any:
            print("[pairs] USE_HIGH_CONF_ONLY=True but no low-confidence columns found; using full analysis mapping")

    out["gene"] = out["gene"].astype(str)
    for c in ("conf_label", "anat_label", "func_label", "plane"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out[out[["gene", "conf_label", "anat_label", "func_label", "plane"]].notna().all(axis=1)].copy()

    out["conf_label"] = out["conf_label"].astype(int)
    out["anat_label"] = out["anat_label"].astype(int)
    out["func_label"] = out["func_label"].astype(int)
    out["plane"] = out["plane"].astype(int)

    out["_sort_dist_func"] = pd.to_numeric(out.get("dist_func_anat_um", np.nan), errors="coerce").fillna(np.inf)
    out["_sort_overlap"] = pd.to_numeric(out.get("overlap_px_func_anat", out.get("overlap_px", np.nan)), errors="coerce").fillna(0)
    out["_sort_dist_conf"] = pd.to_numeric(out.get("dist_conf_anat_um", np.nan), errors="coerce").fillna(np.inf)
    out = out.sort_values(
        ["gene", "anat_label", "_sort_dist_func", "_sort_overlap", "_sort_dist_conf", "plane", "func_label"],
        ascending=[True, True, True, False, True, True, True],
    )
    out = out.drop_duplicates(subset=["gene", "anat_label"], keep="first")
    out = out.drop(columns=["_sort_dist_func", "_sort_overlap", "_sort_dist_conf"], errors="ignore")
    out = out.drop_duplicates(subset=["plane", "func_label", "gene"], keep="first")

    return out.reset_index(drop=True)


def load_midline_bundle(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except Exception:
        return None


def annotate_midline_side(df_in, midline_bundle):
    out = df_in.copy()
    if midline_bundle is None or not isinstance(midline_bundle, dict):
        out["midline_signed_dist_px"] = np.nan
        out["midline_side"] = "unknown"
        out["midline_uncertain"] = False
        return out

    per_plane = midline_bundle.get("per_plane", {})
    if not isinstance(per_plane, dict) or not per_plane:
        out["midline_signed_dist_px"] = np.nan
        out["midline_side"] = "unknown"
        out["midline_uncertain"] = False
        return out

    sides = midline_bundle.get("side_labels", {}) if isinstance(midline_bundle, dict) else {}
    pos_label = str(sides.get("positive", "right")).strip().lower()
    if pos_label not in {"left", "right"}:
        pos_label = "right"
    neg_label = "left" if pos_label == "right" else "right"

    try:
        band = float(midline_bundle.get("manual", {}).get("uncertain_band_px", 0.0))
    except Exception:
        band = 0.0

    d = np.full(len(out), np.nan, dtype=float)

    for p_idx, idx in out.groupby("plane").groups.items():
        p = per_plane.get(int(p_idx), per_plane.get(str(p_idx), None))
        if not isinstance(p, dict):
            continue
        try:
            x0 = float(p.get("x0", np.nan))
            y0 = float(p.get("y0", np.nan))
            th = np.deg2rad(float(p.get("theta_deg", np.nan)))
        except Exception:
            continue
        if not (np.isfinite(x0) and np.isfinite(y0) and np.isfinite(th)):
            continue

        nx = -np.sin(th)
        ny = np.cos(th)
        xv = pd.to_numeric(out.loc[idx, "x"], errors="coerce").to_numpy(dtype=float)
        yv = pd.to_numeric(out.loc[idx, "y"], errors="coerce").to_numpy(dtype=float)
        d[idx] = (xv - x0) * nx + (yv - y0) * ny

    side = np.full(len(out), "unknown", dtype=object)
    finite = np.isfinite(d)
    side[(finite) & (d > band)] = pos_label
    side[(finite) & (d < -band)] = neg_label
    side[(finite) & (np.abs(d) <= band)] = "midline"

    out["midline_signed_dist_px"] = d
    out["midline_side"] = side
    out["midline_uncertain"] = np.abs(d) <= band
    return out


def build_prestim_baseline_windows(df_evt, fps, onset_delay_sec):
    if df_evt is None or getattr(df_evt, "empty", True):
        raise RuntimeError("df_evt is empty")

    evt = df_evt[["event", "time"]].copy()
    evt["event"] = evt["event"].astype(str).str.strip()
    evt["time"] = pd.to_numeric(evt["time"], errors="coerce")
    evt = evt.dropna(subset=["event", "time"]).sort_values("time").reset_index(drop=True)

    stim_starts = {}
    for ev, t in evt[["event", "time"]].itertuples(index=False):
        m = re.match(r"^(B\d+)_stim(\d+)_.+$", str(ev))
        if not m:
            continue
        key = (m.group(1), int(m.group(2)))
        if key not in stim_starts:
            stim_starts[key] = float(t)

    windows = []
    for ev, t_pre in evt[["event", "time"]].itertuples(index=False):
        m = re.match(r"^(B\d+)_prestim(\d+)_pause$", str(ev))
        if not m:
            continue
        key = (m.group(1), int(m.group(2)))
        t_stim = stim_starts.get(key)
        if t_stim is None:
            continue
        t0 = float(t_pre)
        t1 = float(t_stim) + float(onset_delay_sec)
        if not np.isfinite(t0) or not np.isfinite(t1) or t1 <= t0:
            continue
        idx0 = int(round(t0 * float(fps)))
        idx1 = int(round(t1 * float(fps)))
        if idx1 > idx0:
            windows.append((idx0, idx1))

    if not windows:
        raise RuntimeError("no prestim baseline windows found")

    windows = sorted(windows, key=lambda w: (w[0], w[1]))
    merged = []
    for s, e in windows:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    return [(int(s), int(e)) for s, e in merged if e > s]


def compute_zscore_stats(dff, baseline_windows, min_points=200, sigma_eps=1e-6):
    n_roi = int(dff.shape[0])
    T = int(dff.shape[1])
    mask = np.zeros(T, dtype=bool)
    for idx0, idx1 in baseline_windows:
        s = max(0, int(idx0))
        e = min(T, int(idx1))
        if e > s:
            mask[s:e] = True

    mu = np.full(n_roi, np.nan, dtype=np.float32)
    sigma = np.full(n_roi, np.nan, dtype=np.float32)
    n_valid = np.zeros(n_roi, dtype=np.int32)

    if mask.any():
        base = dff[:, mask]
        n_valid = np.isfinite(base).sum(axis=1).astype(np.int32)
        with np.errstate(invalid="ignore", divide="ignore"):
            mu = np.nanmean(base, axis=1).astype(np.float32, copy=False)
            sigma = np.nanstd(base, axis=1).astype(np.float32, copy=False)

    has_points = n_valid >= int(min_points)
    has_sigma = np.isfinite(sigma) & (sigma > float(sigma_eps))
    has_mu = np.isfinite(mu)
    valid = has_points & has_sigma & has_mu
    return {
        "mu": mu,
        "sigma": sigma,
        "n_valid": n_valid,
        "valid": valid,
    }


def extract_window(trace, idx0, idx1, mode="pad_nan", min_valid_points=1):
    T = int(trace.shape[0])
    win_len = int(idx1 - idx0)
    if win_len <= 0:
        return None
    if mode == "strict":
        if idx0 < 0 or idx1 > T:
            return None
        return trace[idx0:idx1]

    seg = np.full(win_len, np.nan, dtype=np.float32)
    src0 = max(int(idx0), 0)
    src1 = min(int(idx1), T)
    if src1 <= src0:
        return None
    dst0 = src0 - int(idx0)
    seg[dst0:dst0 + (src1 - src0)] = trace[src0:src1]
    if (src1 - src0) < max(1, int(min_valid_points)):
        return None
    return seg


def extract_bpi_window(trace, idx0, idx1, mode="pad_nan", min_valid_frac=0.5):
    T = int(trace.shape[0])
    idx0 = int(idx0)
    idx1 = int(idx1)
    win_len = int(idx1 - idx0)
    if win_len <= 0:
        return None, 0, 0

    if mode == "strict":
        if idx0 < 0 or idx1 > T:
            return None, 0, win_len
        seg = trace[idx0:idx1]
        n_valid = int(np.isfinite(seg).sum())
        return seg, n_valid, win_len

    seg = np.full(win_len, np.nan, dtype=np.float32)
    src0 = max(idx0, 0)
    src1 = min(idx1, T)
    if src1 <= src0:
        return None, 0, win_len

    dst0 = src0 - idx0
    seg[dst0:dst0 + (src1 - src0)] = trace[src0:src1]
    n_valid = int(np.isfinite(seg).sum())
    if n_valid < int(np.ceil(float(min_valid_frac) * win_len)):
        return None, n_valid, win_len
    return seg, n_valid, win_len


def parse_stim_components(stype):
    parts = [p.strip().upper() for p in str(stype).split("+") if str(p).strip()]
    if not parts:
        return None
    comps = []
    seen_sides = set()
    for part in parts:
        m = re.match(r"^([LR])(LB|LC)$", part)
        if not m:
            return None
        side, mode = m.group(1), m.group(2)
        if side in seen_sides:
            return None
        seen_sides.add(side)
        comps.append((side, mode))
    return comps


def classify_stim_type(stype):
    parts = [p.strip().upper() for p in str(stype).split("+") if str(p).strip()]
    if not parts:
        return None
    tags = []
    for p in parts:
        if p.endswith("LB"):
            tags.append("B")
        elif p.endswith("LC"):
            tags.append("C")
        else:
            return None
    if all(t == "B" for t in tags):
        return "bout"
    if all(t == "C" for t in tags):
        return "continuous"
    return "mixed"


def combine_segments(arr):
    arr = np.asarray(arr, dtype=np.float32)
    n_valid = np.isfinite(arr).sum(axis=0).astype(np.float32)
    mean = np.divide(
        np.nansum(arr, axis=0),
        n_valid,
        out=np.full(arr.shape[1], np.nan, dtype=np.float32),
        where=n_valid > 0,
    )
    sem = None
    if arr.shape[0] > 1:
        centered = arr - mean[np.newaxis, :]
        centered[~np.isfinite(arr)] = np.nan
        var = np.divide(
            np.nansum(centered * centered, axis=0),
            n_valid - 1.0,
            out=np.full(arr.shape[1], np.nan, dtype=np.float32),
            where=n_valid > 1,
        )
        sem = np.divide(
            np.sqrt(var),
            np.sqrt(n_valid),
            out=np.full(arr.shape[1], np.nan, dtype=np.float32),
            where=n_valid > 1,
        )
    return mean, sem


def resample_segment_to_grid(seg, fps, t_target, window_pre):
    seg = np.asarray(seg, dtype=np.float32)
    t_local = (np.arange(seg.size, dtype=np.float32) / float(fps)) - float(window_pre)
    valid = np.isfinite(seg)
    if int(valid.sum()) < 2:
        return None
    try:
        y = np.interp(
            t_target.astype(np.float64),
            t_local[valid].astype(np.float64),
            seg[valid].astype(np.float64),
            left=np.nan,
            right=np.nan,
        )
        return y.astype(np.float32)
    except Exception:
        return None


In [ ]:
# [cohort-build] Load each fish and aggregate trial/cell/trace data
panel_acc = {p: {} for p in PLOT_ORDER}
all_trial_rows = []
fish_rows = []
mode_durations_cohort = {"LB": [], "LC": []}
mode_event_counts = {"LB": 0, "LC": 0}

cohort_tvec = None
cohort_dt = None if COHORT_TRACE_DT is None else float(COHORT_TRACE_DT)

for spec in FISH_SPECS:
    owner = str(spec["owner"])
    fish_id = str(spec["fish_id"])
    paths = fish_paths(NAS_ROOT, owner, fish_id)
    fish_dir = paths["fish_dir"]
    conf_csv = paths["conf_csv"]
    midline_json = paths["midline_json"]
    suite2p_root = paths["suite2p_root"]
    metadata_dir = paths["metadata_dir"]

    row_status = {
        "owner": owner,
        "fish_id": fish_id,
        "fish_dir": str(fish_dir),
        "ok": False,
        "n_pairs_in": 0,
        "n_pairs_used": 0,
        "n_pairs_with_side": 0,
        "n_trial_rows": 0,
        "n_trace_segments": 0,
        "fps": np.nan,
        "notes": "",
    }

    if not fish_dir.exists():
        row_status["notes"] = "fish_dir missing"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: fish_dir missing -> skip")
        continue
    if not conf_csv.exists():
        row_status["notes"] = "conf_to_func_pairs.csv missing"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: missing {conf_csv} -> skip")
        continue
    if not suite2p_root.exists():
        row_status["notes"] = "suite2P dir missing"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: missing {suite2p_root} -> skip")
        continue

    log_csv = find_experiment_log(metadata_dir, fish_id)
    if log_csv is None or not Path(log_csv).exists():
        row_status["notes"] = "experiment log missing"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: experiment log missing -> skip")
        continue

    polarity = get_polarity(fish_id)

    try:
        df_evt = load_events_df(log_csv, time_scale=STIM_TIME_SCALE)
        df_evt, df_stim, _blocks = build_stim_table(
            df_evt,
            measure_start_block=MEASURE_START_BLOCK,
            measure_start_event=MEASURE_START_EVENT,
            remove_interblock_gaps=REMOVE_INTERBLOCK_GAPS,
        )
    except Exception as e:
        row_status["notes"] = f"stim parsing failed: {e}"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: stim parsing failed -> {e}")
        continue

    if df_stim.empty:
        row_status["notes"] = "df_stim empty"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: df_stim empty -> skip")
        continue

    try:
        s2p_map, roi_df, fps = load_suite2p_map(suite2p_root, polarity)
    except Exception as e:
        row_status["notes"] = f"suite2p load failed: {e}"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: suite2p load failed -> {e}")
        continue

    if not s2p_map:
        row_status["notes"] = "no suite2p planes"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: no suite2p planes -> skip")
        continue
    if fps is None or not np.isfinite(fps) or fps <= 0:
        row_status["notes"] = "invalid fps"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: invalid fps -> skip")
        continue

    row_status["fps"] = float(fps)

    if cohort_tvec is None:
        if cohort_dt is None:
            cohort_dt = 1.0 / float(fps)
        n_pts = int(round((WINDOW_PRE_SEC + WINDOW_POST_SEC) / float(cohort_dt))) + 1
        cohort_tvec = np.linspace(-WINDOW_PRE_SEC, WINDOW_POST_SEC, n_pts, dtype=np.float32)
        print(f"[cohort] cohort_tvec initialized: n={n_pts}, dt={cohort_dt:.6f}s")

    try:
        pairs_raw = pd.read_csv(conf_csv)
    except Exception as e:
        row_status["notes"] = f"failed reading conf csv: {e}"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: failed reading conf csv -> {e}")
        continue

    if "fish_id" in pairs_raw.columns:
        pairs_raw = pairs_raw[pairs_raw["fish_id"].astype(str) == fish_id].copy()

    row_status["n_pairs_in"] = int(len(pairs_raw))

    try:
        pairs = prepare_pairs_for_analysis(pairs_raw)
    except Exception as e:
        row_status["notes"] = f"pair prep failed: {e}"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: pair prep failed -> {e}")
        continue

    if pairs.empty:
        row_status["notes"] = "no valid analysis pairs"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: no valid analysis pairs -> skip")
        continue

    pairs = pairs[pairs["plane"].isin(list(s2p_map.keys()))].copy()
    if pairs.empty:
        row_status["notes"] = "no pairs in loaded suite2p planes"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: no pairs in loaded planes -> skip")
        continue

    pairs = pairs.merge(roi_df, on=["plane", "func_label"], how="left")
    midline_bundle = load_midline_bundle(midline_json)
    pairs = annotate_midline_side(pairs, midline_bundle)
    pairs["midline_side"] = pairs["midline_side"].astype(str).str.strip().str.lower()

    valid_side = pairs["midline_side"].isin({"left", "right"})
    pairs_side = pairs[valid_side].copy()
    row_status["n_pairs_used"] = int(len(pairs))
    row_status["n_pairs_with_side"] = int(len(pairs_side))

    if pairs_side.empty:
        row_status["notes"] = "no pairs with valid midline side"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: no pairs with valid midline side -> skip")
        continue

    try:
        baseline_windows = build_prestim_baseline_windows(df_evt, fps, STIM_ONSET_DELAY_SEC)
    except Exception as e:
        row_status["notes"] = f"baseline windows failed: {e}"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: baseline windows failed -> {e}")
        continue

    zstats_by_plane = {}
    for pidx, pdata in s2p_map.items():
        dff = pdata.get("dff", None)
        if dff is None:
            continue
        zstats_by_plane[int(pidx)] = compute_zscore_stats(
            dff,
            baseline_windows,
            min_points=ZSCORE_MIN_BASELINE_POINTS,
            sigma_eps=ZSCORE_MIN_BASELINE_STD,
        )

    # ---- Traces: unilateral ipsi/contra panels ----
    win_len = int(round((WINDOW_PRE_SEC + WINDOW_POST_SEC) * float(fps))) + 1
    stim_events_uni = []
    for _, r in df_stim.iterrows():
        stype = str(r.get("type", ""))
        comps = parse_stim_components(stype)
        if comps is None or len(comps) != 1:
            continue
        side_letter, mode = comps[0]
        side_word = "left" if side_letter == "L" else "right"
        t0 = float(r.get("start", np.nan)) + float(STIM_ONSET_DELAY_SEC)
        dur = float(r.get("duration", np.nan))
        if not np.isfinite(t0):
            continue
        idx0 = int(round((t0 - WINDOW_PRE_SEC) * float(fps)))
        idx1 = idx0 + win_len
        stim_events_uni.append({
            "side": side_word,
            "mode": mode,
            "idx0": idx0,
            "idx1": idx1,
            "dur": dur,
            "stim_type": stype,
        })
        if mode in mode_durations_cohort and np.isfinite(dur):
            mode_durations_cohort[mode].append(float(dur))
            mode_event_counts[mode] += 1

    if not stim_events_uni:
        row_status["notes"] = "no unilateral stim events"
        fish_rows.append(row_status)
        print(f"[cohort] {fish_id}: no unilateral stim events -> skip traces")
    else:
        n_seg_fish = 0
        genes_present = sorted(pairs_side["gene"].dropna().astype(str).unique().tolist())
        for gene in genes_present:
            sub = pairs_side[pairs_side["gene"].astype(str) == str(gene)]
            for _, pr in sub.iterrows():
                plane = int(pr["plane"])
                roi_idx = int(pr["func_label"]) - 1
                dff = s2p_map[plane].get("dff", None)
                if dff is None or roi_idx < 0 or roi_idx >= dff.shape[0]:
                    continue

                zstats = zstats_by_plane.get(plane, None)
                if zstats is None or roi_idx >= len(zstats["valid"]) or not bool(zstats["valid"][roi_idx]):
                    continue

                trace = (dff[roi_idx] - float(zstats["mu"][roi_idx])) / float(zstats["sigma"][roi_idx])
                cell_side = str(pr.get("midline_side", "unknown")).strip().lower()
                if cell_side not in {"left", "right"}:
                    continue

                roi_key = (fish_id, plane, roi_idx)
                for ev in stim_events_uni:
                    panel = ("ipsi" if ev["side"] == cell_side else "contra") + f"_{ev['mode']}"
                    if panel not in panel_acc:
                        continue
                    seg = extract_window(
                        trace,
                        ev["idx0"],
                        ev["idx1"],
                        mode=WINDOW_EDGE_POLICY,
                        min_valid_points=MIN_VALID_POINTS_PER_SEG,
                    )
                    if seg is None:
                        continue
                    seg_rs = resample_segment_to_grid(seg, fps, cohort_tvec, WINDOW_PRE_SEC)
                    if seg_rs is None:
                        continue
                    gacc = panel_acc[panel].setdefault(gene, {"segs": [], "rois": set(), "source_types": []})
                    gacc["segs"].append(seg_rs)
                    gacc["rois"].add(roi_key)
                    gacc["source_types"].append(ev["stim_type"])
                    n_seg_fish += 1

        row_status["n_trace_segments"] = int(n_seg_fish)

    # ---- BPI trial table (bout/continuous only) ----
    stim_events_bpi = []
    for _, r in df_stim.iterrows():
        stype = str(r.get("type", ""))
        sclass = classify_stim_type(stype)
        if sclass not in {"bout", "continuous"}:
            continue
        t0 = float(r.get("start", np.nan)) + float(STIM_ONSET_DELAY_SEC)
        dur = float(r.get("duration", np.nan))
        if not np.isfinite(t0) or not np.isfinite(dur) or dur <= 0:
            continue
        idx0 = int(round(t0 * float(fps)))
        idx1 = int(round((t0 + dur) * float(fps)))
        if idx1 <= idx0:
            continue
        stim_events_bpi.append({
            "block": r.get("block", np.nan),
            "stim_idx": int(r.get("stim_idx", -1)) if pd.notna(r.get("stim_idx", np.nan)) else -1,
            "stim_type": stype,
            "stim_class": sclass,
            "start_s": t0,
            "duration_s": dur,
            "idx0": idx0,
            "idx1": idx1,
        })

    n_trial_before = len(all_trial_rows)
    if stim_events_bpi:
        for _, pr in pairs_side.iterrows():
            gene = str(pr["gene"])
            anat_label = int(pr["anat_label"])
            plane = int(pr["plane"])
            roi_idx = int(pr["func_label"]) - 1
            dff = s2p_map[plane].get("dff", None)
            if dff is None or roi_idx < 0 or roi_idx >= dff.shape[0]:
                continue
            trace = dff[roi_idx].astype(np.float32, copy=False)

            zstats = zstats_by_plane.get(plane, None)
            trace_z = None
            if zstats is not None and roi_idx < len(zstats["valid"]) and bool(zstats["valid"][roi_idx]):
                trace_z = (trace - float(zstats["mu"][roi_idx])) / float(zstats["sigma"][roi_idx])

            for ev in stim_events_bpi:
                seg, n_valid, n_total = extract_bpi_window(
                    trace,
                    ev["idx0"],
                    ev["idx1"],
                    mode=BPI_EDGE_POLICY,
                    min_valid_frac=BPI_MIN_VALID_FRAC,
                )
                if seg is None:
                    continue
                resp = float(np.nanmean(seg))
                if not np.isfinite(resp):
                    continue

                resp_z = np.nan
                if trace_z is not None:
                    seg_z, _, _ = extract_bpi_window(
                        trace_z,
                        ev["idx0"],
                        ev["idx1"],
                        mode=BPI_EDGE_POLICY,
                        min_valid_frac=BPI_MIN_VALID_FRAC,
                    )
                    if seg_z is not None:
                        _rz = float(np.nanmean(seg_z))
                        if np.isfinite(_rz):
                            resp_z = _rz

                all_trial_rows.append({
                    "owner": owner,
                    "fish_id": fish_id,
                    "gene": gene,
                    "anat_label": anat_label,
                    "plane": plane,
                    "func_label": int(pr["func_label"]),
                    "cell_key": f"{fish_id}|{gene}|{anat_label}",
                    "stim_type": ev["stim_type"],
                    "stim_class": ev["stim_class"],
                    "block": ev["block"],
                    "stim_idx": ev["stim_idx"],
                    "response_mean_dff": resp,
                    "response_mean_zdff": resp_z,
                    "n_valid_points": int(n_valid),
                    "n_total_points": int(n_total),
                    "fps": float(fps),
                })

    row_status["n_trial_rows"] = int(len(all_trial_rows) - n_trial_before)
    row_status["ok"] = True
    if not row_status["notes"]:
        row_status["notes"] = "ok"
    fish_rows.append(row_status)
    print(
        f"[cohort] {fish_id}: pairs={row_status['n_pairs_used']} side_valid={row_status['n_pairs_with_side']} "
        f"segs={row_status['n_trace_segments']} trials={row_status['n_trial_rows']} fps={row_status['fps']:.3f}"
    )

if cohort_tvec is None:
    raise RuntimeError("No valid fish processed; cohort_tvec not initialized.")

fish_summary_df = pd.DataFrame(fish_rows)
globals()["cohort_fish_summary_df"] = fish_summary_df

# Build cohort trace-results dict (panel -> gene -> summary)
cohort_results = {panel: {} for panel in PLOT_ORDER}
for panel in PLOT_ORDER:
    for gene, gacc in panel_acc.get(panel, {}).items():
        segs = gacc.get("segs", [])
        if not segs:
            continue
        arr = np.vstack(segs).astype(np.float32, copy=False)
        mean, sem = combine_segments(arr)
        n_segments = int(arr.shape[0])
        n_cells = int(len(gacc.get("rois", set())))
        n_trials_per_cell = float(n_segments / n_cells) if n_cells > 0 else np.nan
        cohort_results[panel][gene] = {
            "mean": mean,
            "sem": sem,
            "n_segments": n_segments,
            "n_cells": n_cells,
            "n_trials_per_cell": n_trials_per_cell,
            "source_stim_types": sorted(set(gacc.get("source_types", []))),
        }

summary_rows = []
for panel in PLOT_ORDER:
    mode = "LB" if panel.endswith("LB") else "LC"
    n_events_mode = int(mode_event_counts.get(mode, 0))
    for gene, res in cohort_results.get(panel, {}).items():
        summary_rows.append({
            "panel": panel,
            "panel_title": PLOT_TITLES.get(panel, panel),
            "gene": gene,
            "n_cells": int(res.get("n_cells", 0)),
            "n_segments": int(res["n_segments"]),
            "n_trials_per_cell": float(res.get("n_trials_per_cell", np.nan)),
            "n_unilateral_events_mode": n_events_mode,
            "source_stim_types": ", ".join(res.get("source_stim_types", [])),
        })
cohort_summary_df = pd.DataFrame(summary_rows)

trial_df = pd.DataFrame(all_trial_rows)
if trial_df.empty:
    raise RuntimeError("No cohort BPI trial rows were produced.")

# Cell-level BPI table
cell_rows = []
grp = trial_df.groupby(["owner", "fish_id", "gene", "anat_label", "plane", "func_label", "cell_key"], dropna=False)
for keys, sub in grp:
    bout_vals = sub.loc[sub["stim_class"] == "bout", "response_mean_dff"].to_numpy(dtype=float)
    cont_vals = sub.loc[sub["stim_class"] == "continuous", "response_mean_dff"].to_numpy(dtype=float)
    n_b = int(np.isfinite(bout_vals).sum())
    n_c = int(np.isfinite(cont_vals).sum())
    if n_b < BPI_MIN_TRIALS_PER_CLASS or n_c < BPI_MIN_TRIALS_PER_CLASS:
        continue

    b = float(np.nanmean(bout_vals))
    c = float(np.nanmean(cont_vals))
    denom = b + c
    if not np.isfinite(denom) or abs(denom) <= float(BPI_DENOM_EPS):
        continue
    bpi = float((b - c) / denom)
    if not np.isfinite(bpi):
        continue

    bout_vals_z = sub.loc[sub["stim_class"] == "bout", "response_mean_zdff"].to_numpy(dtype=float)
    cont_vals_z = sub.loc[sub["stim_class"] == "continuous", "response_mean_zdff"].to_numpy(dtype=float)
    n_b_z = int(np.isfinite(bout_vals_z).sum())
    n_c_z = int(np.isfinite(cont_vals_z).sum())
    b_z = float(np.nanmean(bout_vals_z)) if n_b_z > 0 else np.nan
    c_z = float(np.nanmean(cont_vals_z)) if n_c_z > 0 else np.nan
    denom_z = b_z + c_z if np.isfinite(b_z) and np.isfinite(c_z) else np.nan
    bpi_z = np.nan
    if n_b_z >= BPI_MIN_TRIALS_PER_CLASS and n_c_z >= BPI_MIN_TRIALS_PER_CLASS:
        if np.isfinite(denom_z) and abs(denom_z) > float(BPI_DENOM_EPS):
            _bpi_z = float((b_z - c_z) / denom_z)
            if np.isfinite(_bpi_z):
                bpi_z = _bpi_z

    owner, fish_id, gene, anat_label, plane, func_label, cell_key = keys
    cell_rows.append({
        "owner": owner,
        "fish_id": fish_id,
        "gene": gene,
        "anat_label": int(anat_label),
        "plane": int(plane),
        "func_label": int(func_label),
        "cell_key": cell_key,
        "n_bout_trials": n_b,
        "n_cont_trials": n_c,
        "n_bout_trials_z": n_b_z,
        "n_cont_trials_z": n_c_z,
        "mean_bout_dff": b,
        "mean_cont_dff": c,
        "mean_bout_zdff": b_z,
        "mean_cont_zdff": c_z,
        "denom": float(denom),
        "denom_z": float(denom_z) if np.isfinite(denom_z) else np.nan,
        "bpi": bpi,
        "bpi_z": bpi_z,
    })

bpi_cells_df = pd.DataFrame(cell_rows)
if bpi_cells_df.empty:
    raise RuntimeError("No cohort cells passed BPI filters.")

# Persist tables for quick reruns
COHORT_OUTDIR.mkdir(parents=True, exist_ok=True)
fish_summary_df.to_csv(COHORT_OUTDIR / "cohort_fish_processing_summary.csv", index=False)
cohort_summary_df.to_csv(COHORT_OUTDIR / "cohort_stim_ipsi_contra_summary.csv", index=False)
trial_df.to_csv(COHORT_OUTDIR / "cohort_bpi_trials.csv", index=False)
bpi_cells_df.to_csv(COHORT_OUTDIR / "cohort_bpi_cells.csv", index=False)

# Expose in-memory globals for plotting cells.
globals()["cohort_tvec"] = cohort_tvec
globals()["cohort_results_stim_ipsi_contra"] = cohort_results
globals()["cohort_mode_durations"] = mode_durations_cohort
globals()["cohort_mode_event_counts"] = mode_event_counts
globals()["cohort_stim_summary_df"] = cohort_summary_df
globals()["cohort_bpi_trials_df"] = trial_df
globals()["cohort_bpi_cells_df"] = bpi_cells_df

print(f"[cohort] processed fish: {int(fish_summary_df['ok'].sum())}/{len(fish_summary_df)}")
print(f"[cohort] bpi cells: {len(bpi_cells_df)}")
print(f"[cohort] trace summary rows: {len(cohort_summary_df)}")
display(fish_summary_df)


In [ ]:
# [56h-cohort] Cohort BPI + condensed ipsi/contra traces
if "cohort_bpi_cells_df" not in globals() or globals().get("cohort_bpi_cells_df") is None or globals()["cohort_bpi_cells_df"].empty:
    raise RuntimeError("cohort_bpi_cells_df missing; run [cohort-build] first.")
if "cohort_results_stim_ipsi_contra" not in globals() or globals().get("cohort_results_stim_ipsi_contra") is None:
    raise RuntimeError("cohort_results_stim_ipsi_contra missing; run [cohort-build] first.")
if "cohort_tvec" not in globals() or globals().get("cohort_tvec") is None:
    raise RuntimeError("cohort_tvec missing; run [cohort-build] first.")

cell_df = globals()["cohort_bpi_cells_df"].copy()
results = globals()["cohort_results_stim_ipsi_contra"]
tvec = np.asarray(globals()["cohort_tvec"], dtype=float)
mode_durations = globals().get("cohort_mode_durations", {"LB": [], "LC": []})
mode_event_counts = globals().get("cohort_mode_event_counts", {"LB": 0, "LC": 0})

TRACE_YMIN = float(globals().get("COMBINED_56H_TRACE_YMIN", -5.0))
TRACE_YMAX = float(globals().get("COMBINED_56H_TRACE_YMAX", 10.0))
FIG_WIDTH = float(globals().get("COHORT_56H_WIDTH", 13.5))
TOP_HEIGHT = float(globals().get("COHORT_56H_TOP_HEIGHT", 4.6))
ROW_HEIGHT = float(globals().get("COHORT_56H_ROW_HEIGHT", 3.6))
HSPACE = float(globals().get("COHORT_56H_HSPACE", 0.36))
WSPACE = float(globals().get("COHORT_56H_WSPACE", 0.20))

PLOT_ORDER = ["contra_LB", "contra_LC", "ipsi_LB", "ipsi_LC"]
PANEL_GRID = [["contra_LB", "contra_LC"], ["ipsi_LB", "ipsi_LC"]]
PLOT_TITLES = {
    "contra_LB": "Contra × bout-like",
    "contra_LC": "Contra × continuous",
    "ipsi_LB": "Ipsi × bout-like",
    "ipsi_LC": "Ipsi × continuous",
}

genes_bpi = set(cell_df["gene"].dropna().astype(str).unique().tolist())
genes_trace = set()
for panel in PLOT_ORDER:
    genes_trace.update(list((results.get(panel, {}) or {}).keys()))
genes_present = genes_bpi | genes_trace
if not genes_present:
    raise RuntimeError("No genes available for cohort [56h] plot.")

genes = [g for g in GENE_ORDER if g in genes_present] + sorted([g for g in genes_present if g not in GENE_ORDER])
gene_colors = {g: GENE_COLORS.get(g, "#666666") for g in genes}

fig = plt.figure(figsize=(FIG_WIDTH, TOP_HEIGHT + 2 * ROW_HEIGHT + 0.4))
gs = fig.add_gridspec(3, 2, height_ratios=[TOP_HEIGHT / ROW_HEIGHT, 1.0, 1.0], hspace=HSPACE, wspace=WSPACE)
ax_bpi = fig.add_subplot(gs[0, :])
axes = np.array([
    [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])],
    [fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])],
])

# Top: BPI distribution by gene
bpi_gene_order = [g for g in genes if g in genes_bpi]
pos = np.arange(len(bpi_gene_order), dtype=float)
bpi_values = [cell_df.loc[cell_df["gene"] == g, "bpi"].to_numpy(dtype=float) for g in bpi_gene_order]

if len(bpi_gene_order) == 0:
    ax_bpi.text(0.5, 0.5, "No BPI data available", transform=ax_bpi.transAxes, ha="center", va="center")
    ax_bpi.set_axis_off()
else:
    bp = ax_bpi.boxplot(
        bpi_values,
        positions=pos,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
    )
    for patch, g in zip(bp["boxes"], bpi_gene_order):
        patch.set_facecolor(gene_colors.get(g, "#bbbbbb"))
        patch.set_alpha(0.35)
        patch.set_edgecolor("black")
    for key in ("whiskers", "caps", "medians"):
        for artist in bp[key]:
            artist.set_color("black")
            artist.set_linewidth(1.0)

    rng = np.random.default_rng(0)
    for x0, vals in zip(pos, bpi_values):
        if vals.size == 0:
            continue
        jitter = rng.uniform(-0.12, 0.12, size=vals.size)
        ax_bpi.scatter(
            np.full(vals.size, x0) + jitter,
            vals,
            s=18,
            c="black",
            alpha=1.0,
            linewidths=0.0,
            zorder=3,
        )

    ax_bpi.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    xtxt = []
    for g in bpi_gene_order:
        n = int((cell_df["gene"] == g).sum())
        xtxt.append(f"{g}(n={n})")
    ax_bpi.set_xticks(pos)
    ax_bpi.set_xticklabels(xtxt)
    ax_bpi.set_ylabel("BPI = (Bout - Continuous) / (Bout + Continuous)")
    ax_bpi.set_ylim(-1.0, 1.0)
    ax_bpi.set_title("Cohort [56e]-style BPI by gene")

# Bottom: 2x2 traces
for r_idx, row_panels in enumerate(PANEL_GRID):
    for c_idx, panel in enumerate(row_panels):
        ax = axes[r_idx, c_idx]
        pdat = results.get(panel, {}) or {}

        gene_items = []
        for g in genes:
            res = pdat.get(g, None)
            if res is None:
                continue
            n_segments = int(res.get("n_segments", 0))
            n_cells = int(res.get("n_cells", 0))
            if n_segments >= MIN_SEGMENTS and n_cells >= MIN_CELLS:
                gene_items.append((g, res))

        if gene_items:
            gene_items = sorted(gene_items, key=lambda kv: int(kv[1].get("n_segments", 0)), reverse=True)[:MAX_GENES]
            for gene, res in gene_items:
                mean = np.asarray(res.get("mean", []), dtype=float)
                sem = res.get("sem", None)
                color = gene_colors.get(gene, None)
                if mean.size != tvec.size:
                    continue
                ax.plot(tvec, mean, label=f"{gene} (n={int(res.get('n_cells', 0))})", color=color)
                if USE_SEM and sem is not None:
                    sem_arr = np.asarray(sem, dtype=float)
                    if sem_arr.size == mean.size:
                        ax.fill_between(tvec, mean - sem_arr, mean + sem_arr, alpha=0.2, color=color)
            ax.legend(fontsize=8, ncol=2)
        else:
            ax.text(0.5, 0.5, "No data after thresholds", transform=ax.transAxes,
                    ha="center", va="center", fontsize=9, alpha=0.8)

        mode = "LB" if panel.endswith("LB") else "LC"
        dur_vals = mode_durations.get(mode, [])
        if isinstance(dur_vals, (list, tuple, np.ndarray)) and len(dur_vals) > 0:
            stim_dur = float(np.median(np.asarray(dur_vals, dtype=float)))
            if np.isfinite(stim_dur) and stim_dur > 0:
                ax.axvspan(0, stim_dur, color="#cccccc", alpha=0.2)

        ax.axvline(0, color="k", linestyle="--", linewidth=1.0)
        ax.axhline(0, color="k", linewidth=0.8, alpha=0.6)
        _n_ev_mode = int(mode_event_counts.get(mode, 0))
        ax.set_title(f"{PLOT_TITLES.get(panel, panel)} (mode events={_n_ev_mode})")

        if r_idx == 1:
            ax.set_xlabel("Time (s)")
        else:
            ax.set_xlabel("")
        if c_idx == 0:
            ax.set_ylabel("z-scored dF/F")
        else:
            ax.set_ylabel("")

        ax.set_ylim(TRACE_YMIN, TRACE_YMAX)

n_fish_ok = int(globals()["cohort_fish_summary_df"].get("ok", pd.Series(dtype=bool)).sum())
fig.suptitle(f"Cohort [56h]: BPI + ipsi/contra traces (fish n={n_fish_ok})", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.98])

out_path = COHORT_OUTDIR / "cohort_56h.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
print(f"[56h-cohort] saved: {out_path}")

globals()["FIG_56H_COHORT_LAST"] = fig
try:
    fig.canvas.draw()
    globals()["FIG_56H_COHORT_RGBA"] = np.asarray(fig.canvas.buffer_rgba()).copy()
except Exception:
    globals()["FIG_56H_COHORT_RGBA"] = None

plt.show()


In [ ]:
# [56g-cohort] Cohort BPI/activity diagnostics (2x2)
if "cohort_bpi_cells_df" not in globals() or globals().get("cohort_bpi_cells_df") is None or globals()["cohort_bpi_cells_df"].empty:
    raise RuntimeError("cohort_bpi_cells_df missing; run [cohort-build] first.")

df = globals()["cohort_bpi_cells_df"].copy()

BPI_ZERO_BAND = float(globals().get('BPI_ZERO_BAND', 0.10))
BPI_NONRESP_Q = float(globals().get('BPI_NONRESP_Q', 0.20))
BPI_NONRESP_ACTIVITY_THRESHOLD = globals().get('BPI_NONRESP_ACTIVITY_THRESHOLD', None)
BPI_POINT_ALPHA = float(globals().get('BPI_POINT_ALPHA', 0.7))
BPI_POINT_SIZE = float(globals().get('BPI_POINT_SIZE', 18.0))
BPI_N_ACTIVITY_BINS = int(globals().get('BPI_N_ACTIVITY_BINS', 6))
BPI_INDEX_COL = str(globals().get('BPI_INDEX_COL', 'bpi')).strip()

if not (0.0 <= BPI_NONRESP_Q <= 1.0):
    print(f"[56g-cohort] invalid BPI_NONRESP_Q={BPI_NONRESP_Q}; using 0.20")
    BPI_NONRESP_Q = 0.20
if BPI_N_ACTIVITY_BINS < 2:
    BPI_N_ACTIVITY_BINS = 2

if {'mean_bout_zdff', 'mean_cont_zdff'}.issubset(df.columns):
    bout_col = 'mean_bout_zdff'
    cont_col = 'mean_cont_zdff'
    activity_label = 'z-scored dF/F'
elif {'mean_bout_dff', 'mean_cont_dff'}.issubset(df.columns):
    bout_col = 'mean_bout_dff'
    cont_col = 'mean_cont_dff'
    activity_label = 'dF/F'
    print('[56g-cohort] WARNING: z-scored columns missing; falling back to raw dF/F.')
else:
    raise RuntimeError('cohort_bpi_cells_df missing response columns needed for [56g].')

if BPI_INDEX_COL not in df.columns:
    fallback_bpi = 'bpi_z' if 'bpi_z' in df.columns else 'bpi'
    print(f"[56g-cohort] BPI_INDEX_COL={BPI_INDEX_COL} missing; using {fallback_bpi}")
    BPI_INDEX_COL = fallback_bpi

for col in [BPI_INDEX_COL, bout_col, cont_col]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['activity_mag'] = (df[bout_col].abs() + df[cont_col].abs()) / 2.0
df['bpi_metric'] = df[BPI_INDEX_COL]
df['abs_bpi'] = df['bpi_metric'].abs()
df = df[
    np.isfinite(df['activity_mag'])
    & np.isfinite(df['bpi_metric'])
    & np.isfinite(df[bout_col])
    & np.isfinite(df[cont_col])
].copy()

if df.empty:
    raise RuntimeError('No finite rows left for [56g-cohort].')

if BPI_NONRESP_ACTIVITY_THRESHOLD is None:
    thr = float(np.nanquantile(df['activity_mag'].to_numpy(dtype=float), BPI_NONRESP_Q))
    thr_label = f"q{int(round(BPI_NONRESP_Q * 100.0))}"
else:
    thr = float(BPI_NONRESP_ACTIVITY_THRESHOLD)
    thr_label = 'manual'

df['is_bpi_near_zero'] = df['bpi_metric'].abs() <= float(BPI_ZERO_BAND)
df['is_nonresponsive'] = df['activity_mag'] <= float(thr) if np.isfinite(thr) else False
df['interpretation'] = np.where(
    df['is_bpi_near_zero'] & df['is_nonresponsive'],
    'near-zero BPI + low activity',
    np.where(
        df['is_bpi_near_zero'] & (~df['is_nonresponsive']),
        'near-zero BPI + high activity',
        'non-zero BPI',
    ),
)

n_total = int(len(df))
n_nz = int(df['is_bpi_near_zero'].sum())
n_nz_low = int((df['is_bpi_near_zero'] & df['is_nonresponsive']).sum())
n_nz_high = int((df['is_bpi_near_zero'] & (~df['is_nonresponsive'])).sum())

print(
    f"[56g-cohort] cells={n_total}; BPI col={BPI_INDEX_COL}; activity cols=({bout_col}, {cont_col}); "
    f"|BPI|<={BPI_ZERO_BAND:.3f}: {n_nz}; near-zero+low={n_nz_low}; near-zero+high={n_nz_high}; "
    f"activity-threshold={thr:.4f} ({thr_label})"
)

genes_present = sorted(df['gene'].dropna().astype(str).unique().tolist())
gene_order = [g for g in GENE_ORDER if g in genes_present] + [g for g in genes_present if g not in GENE_ORDER]

fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
ax_plane = axes[0, 0]
ax_quad = axes[0, 1]
ax_bins = axes[1, 0]
ax_group = axes[1, 1]

# 1) Bout vs Continuous response plane
for gene in gene_order:
    sub = df[df['gene'].astype(str) == str(gene)]
    if sub.empty:
        continue
    ax_plane.scatter(
        sub[cont_col].to_numpy(dtype=float),
        sub[bout_col].to_numpy(dtype=float),
        s=BPI_POINT_SIZE,
        alpha=BPI_POINT_ALPHA,
        color=GENE_COLORS.get(gene, '#777777'),
        edgecolors='none',
        label=str(gene),
    )

sub_nz = df[df['is_bpi_near_zero']]
if not sub_nz.empty:
    ax_plane.scatter(
        sub_nz[cont_col].to_numpy(dtype=float),
        sub_nz[bout_col].to_numpy(dtype=float),
        s=max(14.0, BPI_POINT_SIZE + 8.0),
        facecolors='none',
        edgecolors='black',
        linewidths=0.8,
        alpha=0.8,
        label='|BPI| near zero',
    )

combo = np.concatenate([
    np.abs(df[cont_col].to_numpy(dtype=float)),
    np.abs(df[bout_col].to_numpy(dtype=float)),
])
vmax = float(np.nanpercentile(combo, 99)) if combo.size else 1.0
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 1.0

ax_plane.plot([-vmax, vmax], [-vmax, vmax], linestyle='--', color='black', linewidth=1.0, alpha=0.8)
ax_plane.axhline(0.0, color='#444444', linewidth=0.8)
ax_plane.axvline(0.0, color='#444444', linewidth=0.8)
if np.isfinite(thr):
    diamond = np.array([
        [0.0, 2.0 * thr],
        [2.0 * thr, 0.0],
        [0.0, -2.0 * thr],
        [-2.0 * thr, 0.0],
        [0.0, 2.0 * thr],
    ])
    ax_plane.plot(diamond[:, 0], diamond[:, 1], linestyle=':', color='#222222', linewidth=1.0)
ax_plane.set_xlim(-vmax, vmax)
ax_plane.set_ylim(-vmax, vmax)
ax_plane.set_xlabel(f'Mean continuous response ({activity_label})')
ax_plane.set_ylabel(f'Mean bout response ({activity_label})')
ax_plane.set_title('1) Bout vs Continuous Response Plane')

handles, labels = ax_plane.get_legend_handles_labels()
if handles:
    ax_plane.legend(handles, labels, fontsize=7, ncol=2, loc='lower right')

# 2) BPI vs activity magnitude
for gene in gene_order:
    sub = df[df['gene'].astype(str) == str(gene)]
    if sub.empty:
        continue
    ax_quad.scatter(
        sub['activity_mag'].to_numpy(dtype=float),
        sub['bpi_metric'].to_numpy(dtype=float),
        s=BPI_POINT_SIZE,
        alpha=BPI_POINT_ALPHA,
        color=GENE_COLORS.get(gene, '#777777'),
        edgecolors='none',
    )
if not sub_nz.empty:
    ax_quad.scatter(
        sub_nz['activity_mag'].to_numpy(dtype=float),
        sub_nz['bpi_metric'].to_numpy(dtype=float),
        s=max(14.0, BPI_POINT_SIZE + 8.0),
        facecolors='none',
        edgecolors='black',
        linewidths=0.8,
        alpha=0.8,
    )

ax_quad.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
ax_quad.axhline(float(BPI_ZERO_BAND), color='#444444', linestyle=':', linewidth=1.0)
ax_quad.axhline(-float(BPI_ZERO_BAND), color='#444444', linestyle=':', linewidth=1.0)
if np.isfinite(thr):
    ax_quad.axvline(float(thr), color='#2b2b2b', linestyle='-.', linewidth=1.0)

ax_quad.set_xlabel(f'Activity magnitude = mean(|Bout|, |Continuous|) {activity_label}')
ax_quad.set_ylabel(f'BPI ({BPI_INDEX_COL})')
ax_quad.set_ylim(-1.02, 1.02)
ax_quad.set_xlim(left=0)
ax_quad.set_title('2) BPI vs Activity Magnitude')

txt = (
    f"Near-zero BPI: {n_nz}/{n_total}\n"
    f"low activity: {n_nz_low}\n"
    f"higher activity: {n_nz_high}"
)
ax_quad.text(
    0.02,
    0.98,
    txt,
    transform=ax_quad.transAxes,
    ha='left',
    va='top',
    fontsize=8.5,
    bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.85),
)

# 3) Activity-binned tuning strength
binned_df = pd.DataFrame()
n_bins = min(BPI_N_ACTIVITY_BINS, int(df['activity_mag'].nunique()))
if n_bins >= 2:
    try:
        bin_cat = pd.qcut(df['activity_mag'], q=n_bins, duplicates='drop')
        tmp = df.copy()
        tmp['_activity_bin'] = bin_cat
        rows = []
        for b, sub in tmp.groupby('_activity_bin', observed=False):
            vals = sub['abs_bpi'].to_numpy(dtype=float)
            if vals.size == 0:
                continue
            rows.append({
                'bin': str(b),
                'bin_lo': float(b.left),
                'bin_hi': float(b.right),
                'bin_mid': float((b.left + b.right) / 2.0),
                'n_cells': int(vals.size),
                'median_abs_bpi': float(np.nanmedian(vals)),
                'q25_abs_bpi': float(np.nanquantile(vals, 0.25)),
                'q75_abs_bpi': float(np.nanquantile(vals, 0.75)),
            })
        binned_df = pd.DataFrame(rows).sort_values('bin_mid').reset_index(drop=True)
    except Exception:
        binned_df = pd.DataFrame()

if binned_df.empty:
    ax_bins.text(0.5, 0.5, 'Insufficient data for binning', transform=ax_bins.transAxes, ha='center', va='center')
    ax_bins.set_title('3) Activity-Binned |BPI|')
    ax_bins.set_xlabel(f'Activity magnitude ({activity_label})')
    ax_bins.set_ylabel('Median |BPI|')
else:
    x = binned_df['bin_mid'].to_numpy(dtype=float)
    y = binned_df['median_abs_bpi'].to_numpy(dtype=float)
    y25 = binned_df['q25_abs_bpi'].to_numpy(dtype=float)
    y75 = binned_df['q75_abs_bpi'].to_numpy(dtype=float)
    ax_bins.plot(x, y, marker='o', color='#1f77b4', linewidth=1.8)
    ax_bins.fill_between(x, y25, y75, alpha=0.2, color='#1f77b4')
    for _, r in binned_df.iterrows():
        ax_bins.text(float(r['bin_mid']), float(r['median_abs_bpi']) + 0.015, f"n={int(r['n_cells'])}", ha='center', va='bottom', fontsize=7)
    rho = pd.Series(df['activity_mag']).corr(pd.Series(df['abs_bpi']), method='spearman')
    if np.isfinite(rho):
        ax_bins.text(0.02, 0.98, f"Spearman ρ={rho:.2f}", transform=ax_bins.transAxes, ha='left', va='top', fontsize=8.5)
    ax_bins.set_title('3) Activity-Binned Tuning Strength')
    ax_bins.set_xlabel(f'Activity magnitude ({activity_label}; bin mid)')
    ax_bins.set_ylabel('Median |BPI|')
    ax_bins.set_ylim(0, 1.02)
    ax_bins.set_xlim(left=0)

# 4) Activity by BPI/response class
group_order = [
    'near-zero BPI + low activity',
    'near-zero BPI + high activity',
    'non-zero BPI',
]
group_colors = {
    'near-zero BPI + low activity': '#9e9e9e',
    'near-zero BPI + high activity': '#5ab4ac',
    'non-zero BPI': '#f46d43',
}

group_data = []
group_labels = []
for g in group_order:
    vals = df.loc[df['interpretation'] == g, 'activity_mag'].to_numpy(dtype=float)
    if vals.size == 0:
        continue
    group_data.append(vals)
    group_labels.append(f"{g}\n(n={vals.size})")

if not group_data:
    ax_group.text(0.5, 0.5, 'No group data', transform=ax_group.transAxes, ha='center', va='center')
    ax_group.set_title('4) Activity by BPI/Response Class')
    ax_group.set_ylabel(f'Activity magnitude ({activity_label})')
else:
    bp = ax_group.boxplot(group_data, patch_artist=True, showfliers=False)
    for i, patch in enumerate(bp['boxes']):
        base = group_labels[i].split('\n')[0]
        patch.set_facecolor(group_colors.get(base, '#bbbbbb'))
        patch.set_alpha(0.45)
        patch.set_edgecolor('#333333')
    for key in ('whiskers', 'caps', 'medians'):
        for artist in bp[key]:
            artist.set_color('#333333')
            artist.set_linewidth(1.0)

    rng = np.random.default_rng(0)
    for i, vals in enumerate(group_data, start=1):
        jitter = rng.uniform(-0.10, 0.10, size=vals.size)
        ax_group.scatter(
            np.full(vals.size, i) + jitter,
            vals,
            s=10,
            alpha=0.35,
            color='#222222',
            edgecolors='none',
        )

    ax_group.set_xticks(np.arange(1, len(group_labels) + 1))
    ax_group.set_xticklabels(group_labels, rotation=12, ha='right')
    ax_group.set_ylabel(f'Activity magnitude ({activity_label})')
    ax_group.set_title('4) Activity by BPI/Response Class')
    if np.isfinite(thr):
        ax_group.axhline(float(thr), color='#2b2b2b', linestyle='-.', linewidth=1.0)

n_fish_ok = int(globals()['cohort_fish_summary_df'].get('ok', pd.Series(dtype=bool)).sum())
fig.suptitle(f'Cohort [56g]: BPI vs activity diagnostics (fish n={n_fish_ok})', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])

out_path = COHORT_OUTDIR / 'cohort_56g.png'
fig.savefig(out_path, dpi=200, bbox_inches='tight')
print(f"[56g-cohort] saved: {out_path}")

globals()['cohort_bpi_activity_df'] = df.copy()
globals()['cohort_bpi_activity_bins_df'] = binned_df.copy()
globals()['FIG_56G_COHORT_LAST'] = fig
try:
    fig.canvas.draw()
    globals()['FIG_56G_COHORT_RGBA'] = np.asarray(fig.canvas.buffer_rgba()).copy()
except Exception:
    globals()['FIG_56G_COHORT_RGBA'] = None

try:
    breakdown = (
        df.groupby(['gene', 'interpretation'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    globals()['cohort_bpi_activity_breakdown_df'] = breakdown.copy()
    display(breakdown)
except Exception:
    pass

plt.show()
